# Phase 2 — Détection d'anomalies réseau avec CIC-IDS2017

**Projet : Système de détection d’anomalies basé sur l’intelligence artificielle**

Ce notebook construit un premier modèle de détection d'anomalies réseau avec **Isolation Forest**.

## Objectif

1. Charger CIC-IDS2017 depuis Hugging Face.
2. Explorer les flux réseau et leurs étiquettes (`Label`).
3. Nettoyer les données et sélectionner les caractéristiques numériques.
4. Entraîner Isolation Forest uniquement sur des flux `BENIGN`.
5. Évaluer sa capacité à signaler les flux d'attaque comme anomalies.
6. Exporter le modèle pour son utilisation ultérieure dans le backend FastAPI.

> **Choix méthodologique :** les étiquettes ne sont pas données au modèle pendant l'entraînement. Elles servent uniquement à isoler le trafic normal pour l'apprentissage et à mesurer les résultats après prédiction. C'est une démarche de détection d'anomalies non supervisée (one-class).

## 0. Environnement d'exécution

Exécuter ce notebook dans **Google Colab**. Dans le menu *Exécution → Modifier le type d'exécution*, choisir si possible un environnement avec suffisamment de mémoire RAM. Un GPU n'est pas nécessaire pour Isolation Forest.

Le dépôt Hugging Face annonce environ **2,83 millions de lignes** et une taille totale d'environ **885 Mo**. Pour éviter de saturer la RAM et de rendre l'entraînement trop long, nous travaillerons d'abord sur un échantillon configurable. Le code conserve la possibilité d'augmenter progressivement cette taille.

In [ ]:
!pip -q install -U datasets pandas numpy scikit-learn matplotlib seaborn joblib

## 1. Importer les bibliothèques

- `datasets` : télécharger le dataset depuis Hugging Face.
- `pandas` et `numpy` : manipuler les données.
- `scikit-learn` : préparer les données, entraîner et évaluer le modèle.
- `matplotlib` et `seaborn` : visualiser les résultats.
- `joblib` : sauvegarder le modèle entraîné.

In [ ]:
import json
import os
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    roc_auc_score,
    average_precision_score
)

warnings.filterwarnings("ignore")
RANDOM_STATE = 42

print("Bibliothèques importées.")

## 2. Télécharger CIC-IDS2017 depuis Hugging Face

Nous utilisons le chargement demandé dans le sujet :

```python
from datasets import load_dataset
ds = load_dataset("c01dsnap/CIC-IDS2017")
```

Le dépôt expose un split `train`. Il s'agit ici du nom du split du dépôt, pas encore de notre séparation entraînement/test. Nous créerons notre propre séparation pour évaluer le modèle.

In [ ]:
from datasets import load_dataset

ds = load_dataset("c01dsnap/CIC-IDS2017")

print(ds)
print("\\nSplits disponibles :", list(ds.keys()))
print("Nombre de lignes :", len(ds["train"]))
print("Colonnes disponibles :", ds["train"].column_names)

## 3. Construire un échantillon exploitable

La conversion de plusieurs millions de lignes en DataFrame peut consommer beaucoup de RAM. Nous mélangeons donc le split puis sélectionnons un échantillon.

- `MAX_ROWS` : nombre maximum de lignes chargées dans Pandas.
- Commencer avec 300 000 à 500 000 lignes dans Colab.
- Augmenter cette valeur uniquement si la mémoire disponible le permet.

L'échantillonnage est effectué avant la conversion en DataFrame pour limiter l'utilisation de la mémoire.

In [ ]:
MAX_ROWS = 500_000  # Réduire à 200_000 si la RAM de Colab est limitée

hf_train = ds["train"]
n_rows = min(MAX_ROWS, len(hf_train))

sampled_dataset = hf_train.shuffle(seed=RANDOM_STATE).select(range(n_rows))
df = sampled_dataset.to_pandas()

print("Dimensions de l'échantillon :", df.shape)
display(df.head())

## 4. Explorer le dataset

Chaque ligne représente un **flux réseau** décrit par des caractéristiques calculées à partir des paquets. Ce n'est donc pas un paquet isolé capturé en direct par Scapy.

La colonne `Label` indique la classe réelle du flux : `BENIGN` pour le trafic normal et différents noms pour les activités d'attaque.

In [ ]:
df.info()

In [ ]:
print("Nombre de lignes et de colonnes :", df.shape)
print("\\nValeurs de la colonne Label :")
display(df["Label"].astype(str).str.strip().value_counts().to_frame("nombre"))

print("\\nStatistiques descriptives :")
display(df.describe(include="all").T.head(25))

In [ ]:
label_counts = df["Label"].astype(str).str.strip().value_counts()

plt.figure(figsize=(12, 5))
sns.barplot(x=label_counts.index, y=label_counts.values)
plt.title("Répartition des labels dans l'échantillon CIC-IDS2017")
plt.xlabel("Label")
plt.ylabel("Nombre de flux")
plt.xticks(rotation=75, ha="right")
plt.tight_layout()
plt.show()

### Interprétation

Cette exploration permet de voir si l'échantillon contient suffisamment de flux normaux et de flux d'attaque. Les attaques peuvent être réparties entre plusieurs catégories (`DoS`, `DDoS`, `PortScan`, etc.).

Pour notre première expérience, nous allons ramener ces catégories à deux groupes uniquement pour l'évaluation :
- `0` : trafic normal (`BENIGN`)
- `1` : trafic non-BENIGN (attaque / trafic malveillant étiqueté dans le dataset)

Cette conversion binaire est utilisée pour mesurer la détection, pas pour entraîner un classifieur supervisé.

## 5. Nettoyer et préparer les données

Les datasets réels peuvent contenir :
- des noms de colonnes avec des espaces ;
- des valeurs infinies (`inf`, `-inf`) ;
- des valeurs manquantes (`NaN`) ;
- des colonnes non numériques ;
- des lignes dupliquées.

Nous supprimons les doublons exacts et les lignes sans label. Nous ne supprimons pas automatiquement les valeurs extrêmes : elles peuvent justement représenter des comportements anormaux.

In [ ]:
# Nettoyage léger des noms de colonnes et des labels
df.columns = df.columns.astype(str).str.strip()
df["Label"] = df["Label"].astype(str).str.strip()

# Retirer les lignes sans étiquette exploitable
df = df[df["Label"].notna() & (df["Label"] != "")].copy()

# Supprimer les doublons exacts
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Doublons supprimés : {before - len(df):,}")

# Créer la cible binaire pour l'évaluation seulement
df["target"] = (df["Label"].str.upper() != "BENIGN").astype("int8")

print("\\nRépartition binaire :")
display(df["target"].value_counts().rename(index={0: "Normal", 1: "Attaque"}).to_frame("nombre"))

### 5.1 Séparer les données avant d'apprendre les transformations

Nous séparons l'échantillon en deux parties :
- une partie de développement ;
- une partie de test final.

La séparation est stratifiée pour conserver approximativement la proportion de trafic normal et d'attaque dans les deux parties.

**Important :** le jeu de test ne sera jamais utilisé pour entraîner le modèle ni pour calculer les médianes de remplacement des valeurs manquantes.

In [ ]:
development_df, test_df = train_test_split(
    df,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=df["target"]
)

print("Développement :", development_df.shape)
print("Test :", test_df.shape)
print("\\nProportions dans le test :")
display(test_df["target"].value_counts(normalize=True).rename(index={0: "Normal", 1: "Attaque"}))

## 6. Sélectionner les caractéristiques (features)

Le modèle ne doit pas recevoir les colonnes `Label` ou `target`, car elles contiennent la réponse attendue. Les adresses IP ne sont pas présentes dans la table de caractéristiques affichée par ce dépôt ; nous retenons ici les colonnes numériques du flux.

Nous :
1. utilisons les colonnes numériques ;
2. retirons les colonnes entièrement vides dans les données normales d'entraînement ;
3. retirons les colonnes constantes dans ces mêmes données ;
4. conservons exactement le même ordre de colonnes pour le test.

Le choix des features pourra être amélioré après une première expérience.

In [ ]:
# L'algorithme apprend uniquement à partir des flux BENIGN
normal_train_df = development_df[
    development_df["Label"].str.upper() == "BENIGN"
].copy()

# Colonnes numériques, sans les labels
numeric_columns = development_df.select_dtypes(include=[np.number]).columns.tolist()
numeric_columns = [c for c in numeric_columns if c not in ["target"]]

# Matrice d'entraînement composée uniquement de trafic normal
X_normal_raw = normal_train_df[numeric_columns].replace([np.inf, -np.inf], np.nan)

# Garder les colonnes qui ont au moins une valeur et plus d'une valeur distincte
non_empty_columns = X_normal_raw.columns[X_normal_raw.notna().any()].tolist()
X_normal_raw = X_normal_raw[non_empty_columns]

varying_columns = [
    c for c in X_normal_raw.columns
    if X_normal_raw[c].nunique(dropna=True) > 1
]

feature_columns = varying_columns
X_normal = X_normal_raw[feature_columns]

# Données de test : mêmes colonnes et même ordre
X_test = (
    test_df[feature_columns]
    .replace([np.inf, -np.inf], np.nan)
)

y_test = test_df["target"].astype(int)

print("Nombre de features retenues :", len(feature_columns))
print("Nombre de flux normaux pour l'entraînement :", len(X_normal))
print("Nombre de flux pour le test :", len(X_test))
print("\\nPremières features :")
print(feature_columns[:20])

## 7. Limiter la taille de l'entraînement si nécessaire

Isolation Forest peut traiter de nombreux exemples, mais l'entraînement sur un très grand nombre de lignes augmente le temps de calcul.

Nous limitons donc le nombre de flux normaux d'entraînement. Le test reste inchangé afin de conserver une évaluation indépendante.

In [ ]:
MAX_TRAIN_NORMAL = 150_000  # Augmenter progressivement si les ressources le permettent

if len(X_normal) > MAX_TRAIN_NORMAL:
    X_train = X_normal.sample(
        n=MAX_TRAIN_NORMAL,
        random_state=RANDOM_STATE
    )
else:
    X_train = X_normal.copy()

print("Taille finale de X_train :", X_train.shape)
print("Taille de X_test :", X_test.shape)

## 8. Construire et entraîner Isolation Forest

### Rappel du principe

Isolation Forest construit des arbres qui isolent les observations. Les points atypiques ont tendance à être isolés plus rapidement que les observations habituelles.

Paramètres principaux :
- `n_estimators=200` : nombre d'arbres ;
- `contamination="auto"` : seuil interne automatique de l'algorithme ;
- `random_state=42` : rendre l'expérience reproductible ;
- `n_jobs=-1` : utiliser les cœurs CPU disponibles.

Nous plaçons une imputation médiane dans un `Pipeline`. Les médianes sont calculées uniquement sur les données d'entraînement normales.

In [ ]:
model_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("isolation_forest", IsolationForest(
        n_estimators=200,
        contamination="auto",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

model_pipeline.fit(X_train)

print("Entraînement terminé.")

## 9. Effectuer les prédictions

La méthode `predict()` d'Isolation Forest renvoie :
- `1` : observation considérée comme normale ;
- `-1` : observation considérée comme anomalie.

Pour comparer avec les labels réels, nous convertissons ces résultats :
- prédiction `1` devient `0` (normal) ;
- prédiction `-1` devient `1` (anomalie).

Le modèle ne reçoit pas `y_test` pour prédire.

In [ ]:
raw_predictions = model_pipeline.predict(X_test)

# Convertir la sortie IF en classes binaires : 0 normal, 1 anomalie
y_pred = np.where(raw_predictions == -1, 1, 0)

print("Nombre de flux testés :", len(y_pred))
print("Anomalies détectées :", int((y_pred == 1).sum()))
print("Flux considérés normaux :", int((y_pred == 0).sum()))

## 10. Évaluer le modèle

Nous comparons les prédictions aux labels réels du dataset.

Métriques :
- **Precision** : parmi les alertes émises, quelle proportion correspond réellement à une attaque ?
- **Recall** : parmi les attaques présentes, quelle proportion a été détectée ?
- **F1-score** : équilibre entre precision et recall.
- **Accuracy** : proportion globale de prédictions correctes.

Dans un système de détection d'intrusion, precision et recall sont particulièrement importantes. Une forte quantité de fausses alertes peut rendre le système inutilisable.

In [ ]:
print(classification_report(
    y_test,
    y_pred,
    labels=[0, 1],
    target_names=["Normal", "Attaque"],
    zero_division=0
))

metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "precision_attack": precision_score(y_test, y_pred, zero_division=0),
    "recall_attack": recall_score(y_test, y_pred, zero_division=0),
    "f1_attack": f1_score(y_test, y_pred, zero_division=0)
}

try:
    anomaly_scores = -model_pipeline.decision_function(X_test)
    metrics["roc_auc"] = roc_auc_score(y_test, anomaly_scores)
    metrics["average_precision"] = average_precision_score(y_test, anomaly_scores)
except ValueError as error:
    print("Certaines métriques de score ne peuvent pas être calculées :", error)

display(pd.DataFrame([metrics]).T.rename(columns={0: "score"}))

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Prédit normal", "Prédit anomalie"],
    yticklabels=["Réel normal", "Réel attaque"]
)
plt.title("Matrice de confusion — Isolation Forest")
plt.xlabel("Prédiction")
plt.ylabel("Valeur réelle")
plt.tight_layout()
plt.show()

### Comment lire la matrice de confusion ?

- **Vrai négatif (TN)** : un flux normal correctement reconnu comme normal.
- **Faux positif (FP)** : un flux normal signalé à tort comme anomalie.
- **Faux négatif (FN)** : une attaque qui n'a pas été détectée.
- **Vrai positif (TP)** : une attaque correctement signalée comme anomalie.

Un modèle intéressant doit détecter les attaques tout en limitant les faux positifs. Les résultats dépendent de l'échantillon, des features, des paramètres et de la qualité des labels.

## 11. Examiner le score d'anomalie

`decision_function()` produit un score de décision. Pour faciliter la lecture, nous prenons son opposé : plus ce score transformé est élevé, plus l'observation est atypique selon le modèle.

Ce score peut être utile plus tard pour afficher un indicateur de suspicion dans le dashboard. Il ne représente pas une probabilité d'attaque.

In [ ]:
anomaly_scores = -model_pipeline.decision_function(X_test)

results_df = pd.DataFrame({
    "score_anomalie": anomaly_scores,
    "label_reel": y_test.to_numpy(),
    "prediction": y_pred
})

plt.figure(figsize=(9, 5))
sns.histplot(
    data=results_df,
    x="score_anomalie",
    hue="label_reel",
    bins=60,
    stat="density",
    common_norm=False,
    element="step"
)
plt.title("Distribution des scores d'anomalie")
plt.xlabel("Score d'anomalie (plus élevé = plus atypique)")
plt.tight_layout()
plt.show()

display(results_df.head(10))

## 12. (Optionnel) Tester One-Class SVM

Le sujet mentionne aussi SVM. Pour la détection non supervisée d'une seule classe, on utilise **One-Class SVM**.

Attention : One-Class SVM peut être beaucoup plus lent qu'Isolation Forest sur de grands volumes. Cette cellule utilise donc un petit sous-échantillon. Elle est facultative et sert à une comparaison expérimentale, pas à remplacer le modèle principal.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM

RUN_OCSVM = False  # Passer à True pour lancer l'expérience facultative

if RUN_OCSVM:
    svm_train_n = min(10_000, len(X_train))
    svm_test_n = min(10_000, len(X_test))

    X_svm_train = X_train.sample(n=svm_train_n, random_state=RANDOM_STATE)
    X_svm_test = X_test.sample(n=svm_test_n, random_state=RANDOM_STATE)
    y_svm_test = y_test.loc[X_svm_test.index]

    svm_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("one_class_svm", OneClassSVM(kernel="rbf", nu=0.05, gamma="scale"))
    ])

    svm_pipeline.fit(X_svm_train)
    svm_raw = svm_pipeline.predict(X_svm_test)
    svm_pred = np.where(svm_raw == -1, 1, 0)

    print(classification_report(
        y_svm_test,
        svm_pred,
        labels=[0, 1],
        target_names=["Normal", "Attaque"],
        zero_division=0
    ))
else:
    print("One-Class SVM ignoré. Mettre RUN_OCSVM = True pour lancer cette comparaison.")

## 13. Exporter le modèle pour FastAPI

Nous sauvegardons :
- `isolation_forest_pipeline.joblib` : pipeline complet (imputation + modèle) ;
- `feature_columns.json` : noms et ordre des caractéristiques attendues ;
- `model_metadata.json` : informations utiles sur l'expérience.

**Important pour l'intégration :** les données envoyées à ce modèle devront contenir les mêmes caractéristiques que celles utilisées ici, avec les mêmes unités et le même mode de calcul. Le modèle CIC-IDS2017 est entraîné sur des **flux réseau** (features de type CICFlowMeter), pas directement sur des paquets bruts Scapy. Il faudra donc construire un extracteur de flux compatible côté application avant de lui transmettre les données.

In [ ]:
OUTPUT_DIR = Path("artifacts")
OUTPUT_DIR.mkdir(exist_ok=True)

model_path = OUTPUT_DIR / "isolation_forest_pipeline.joblib"
features_path = OUTPUT_DIR / "feature_columns.json"
metadata_path = OUTPUT_DIR / "model_metadata.json"

joblib.dump(model_pipeline, model_path)

with open(features_path, "w", encoding="utf-8") as file:
    json.dump(feature_columns, file, ensure_ascii=False, indent=2)

metadata = {
    "model_name": "IsolationForest",
    "dataset": "c01dsnap/CIC-IDS2017",
    "training_strategy": "unsupervised_one_class_benign_only",
    "normal_label": "BENIGN",
    "anomaly_output": -1,
    "normal_output": 1,
    "n_features": len(feature_columns),
    "n_train_normal": int(len(X_train)),
    "n_test": int(len(X_test)),
    "random_state": RANDOM_STATE,
    "metrics": {k: float(v) for k, v in metrics.items()}
}

with open(metadata_path, "w", encoding="utf-8") as file:
    json.dump(metadata, file, ensure_ascii=False, indent=2)

print("Fichiers exportés :")
print(model_path)
print(features_path)
print(metadata_path)

### 13.1 Vérifier le chargement du modèle sauvegardé

Cette vérification simule le futur chargement dans le backend FastAPI.

In [ ]:
loaded_model = joblib.load(model_path)

with open(features_path, "r", encoding="utf-8") as file:
    loaded_features = json.load(file)

# Prendre quelques lignes du test pour vérifier le format d'entrée
sample_input = X_test[loaded_features].head(5)
sample_output = loaded_model.predict(sample_input)

print("Prédictions brutes :", sample_output)
print("Interprétation :", ["Anomalie" if p == -1 else "Normal" for p in sample_output])

### 13.2 Télécharger les artefacts depuis Colab

Le modèle et ses fichiers de métadonnées sont regroupés dans une archive ZIP téléchargeable.

In [ ]:
import shutil

archive_path = shutil.make_archive("cic_ids2017_model_artifacts", "zip", OUTPUT_DIR)
print("Archive créée :", archive_path)

try:
    from google.colab import files
    files.download(archive_path)
except ImportError:
    print("Environnement hors Colab : récupérer le dossier artifacts manuellement.")

## 14. Conclusion et prochaines améliorations

À la fin de cette phase, nous disposons d'un premier modèle entraîné sur le trafic normal de CIC-IDS2017 et évalué sur un mélange de trafic normal et d'attaques.

### Livrables

- Un modèle `isolation_forest_pipeline.joblib`.
- La liste ordonnée des caractéristiques `feature_columns.json`.
- Les informations et métriques `model_metadata.json`.
- Une évaluation avec rapport de classification et matrice de confusion.

### Limites à mentionner dans le rapport

1. CIC-IDS2017 contient des flux réseau pré-calculés. Il ne s'agit pas d'une capture en direct.
2. Une anomalie statistique n'est pas nécessairement une attaque.
3. Les performances sur un échantillon aléatoire ne garantissent pas les mêmes résultats sur un réseau réel.
4. Le choix de `contamination`, des features et de la taille de l'échantillon influence les résultats.
5. Il faudra vérifier la compatibilité entre les features CICFlowMeter et les caractéristiques produites par notre futur extracteur de flux.

### Étape suivante

Dans la phase backend, FastAPI chargera le pipeline sauvegardé, recevra un vecteur de caractéristiques de flux réseau, retournera la prédiction et enregistrera les alertes. La capture Scapy devra être reliée à un calcul de flux qui produit les mêmes colonnes que le dataset.